### Data Ingestion

In [139]:
from langchain_core.documents import Document

In [140]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source.txt", "pages": 1,
        "author": "John Doe", 
        "date_created": "2024-06-01"
    },
)
doc

Document(metadata={'source': 'test_source.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is a test document.')

In [141]:
## Create a simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [142]:
sample_texts = {
    # Path to Python language documentation file
    "../data/text_files/python.txt": """Python is a high-level, interpreted programming language.
- Easy to learn and read with clean syntax
- Versatile: web development, data science, AI, automation
- Extensive standard library and third-party packages
- Dynamic typing with strong type inference
- Cross-platform support (Windows, Linux, macOS)
- Large community and excellent documentation
- Supports multiple programming paradigms""",
    # Path to C language documentation file
    "../data/text_files/c.txt": """C is a low-level, compiled programming language.
- Provides direct memory access and system-level control
- Foundation for many modern languages (C++, Java, Python)
- Highly efficient and fast execution
- Requires manual memory management
- Portable across different hardware platforms
- Used in operating systems, embedded systems, drivers
- Rich set of operators and control structures""",
}

In [143]:
for filepath, content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [144]:
# TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python.txt", encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]


In [145]:
# DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    "../data/text_files", 
    glob="*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={"encoding": "utf-8"}, 
    show_progress=False
)
documents = loader.load()
print(documents)    

[Document(metadata={'source': '../data/text_files/c.txt'}, page_content='C is a low-level, compiled programming language.\n- Provides direct memory access and system-level control\n- Foundation for many modern languages (C++, Java, Python)\n- Highly efficient and fast execution\n- Requires manual memory management\n- Portable across different hardware platforms\n- Used in operating systems, embedded systems, drivers\n- Rich set of operators and control structures'), Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms'), Document(metadata={'source': '../data/text_file

### Chunking

In [146]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [147]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [148]:
chunks=split_documents(documents)
chunks

Split 3 documents into 4 chunks

Example chunk:
Content: C is a low-level, compiled programming language.
- Provides direct memory access and system-level control
- Foundation for many modern languages (C++, Java, Python)
- Highly efficient and fast executi...
Metadata: {'source': '../data/text_files/c.txt'}


[Document(metadata={'source': '../data/text_files/c.txt'}, page_content='C is a low-level, compiled programming language.\n- Provides direct memory access and system-level control\n- Foundation for many modern languages (C++, Java, Python)\n- Highly efficient and fast execution\n- Requires manual memory management\n- Portable across different hardware platforms\n- Used in operating systems, embedded systems, drivers\n- Rich set of operators and control structures'),
 Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms'),
 Document(metadata={'source': '../data/text_fi

### Embedding

In [149]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [150]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded.")
                             
        print(f"Generating embeddings for {len(text)} texts...")
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings

In [151]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [152]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection of PDF document embeddings"})

            print(f"Vector store initialized successfully. Collection name: '{self.collection_name}' Existing document at collection '{self.collection.count()}'")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        # Prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embedding_list
            )
            print(f"Added {len(documents)} documents and embeddings to the vector store successfully.")
        except Exception as e:
            print(f"Error adding embeddings to vector store: {e}")
            raise

In [153]:
vector_store = VectorStore()
vector_store

Vector store initialized successfully. Collection name: 'pdf_documents' Existing document at collection '12'


### Convert the texts to embeddings

In [154]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 4 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.


### Store embeddings into vector store

In [155]:
vector_store.add_documents(chunks, embeddings)

Added 4 documents and embeddings to the vector store successfully.


### Rag Retriever

In [156]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        print(f"Retrieving relevant documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query: '{query}'")
                return retrieved_docs
            
            else:
                print(f"Failed to retrieve results for query: '{query}'")


        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise

In [157]:
reg_retriever = RAGRetriever(vector_store, embedding_manager)
reg_retriever

### Query

In [158]:
reg_retriever.retrieve("What are the key feature of python?")

Retrieving relevant documents for query: 'What are the key feature of python?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
Retrieved 5 relevant documents for the query: 'What are the key feature of python?'


[{'id': 'doc_7df8ddfa_1',
  'content': 'Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms',
  'metadata': {'doc_index': 1,
   'content_length': 394,
   'source': '../data/text_files/python.txt'},
  'similarity_score': 0.26044291257858276,
  'distance': 0.7395570874214172,
  'rank': 1},
 {'id': 'doc_e3582611_1',
  'content': 'Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large 

### Integration VectorDB Context Pipeline with LLM Output

In [159]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Force reload of environment variables
load_dotenv('../.env.local', override=True)

## Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")
print(f"API Key loaded: {groq_api_key[:10]}..." if groq_api_key else "API Key not found!")

llm = ChatGroq(api_key=groq_api_key, model_name='llama-3.1-8b-instant', temperature=0.1, max_tokens=500)


API Key loaded: gsk_Q0Zq2e...


In [160]:
def rag_simple(query: str, retriever, llm, top_K = 3):
    results = retriever.retrieve(query, top_k=top_K)

    context = "\n\n".join(doc['content'] for doc in results) if results else ""

    print(context)

    if not context:
        return "No relevant information found in the documents."
    
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = llm.invoke([prompt.format(context=context, query=query)])

    return response.content

In [161]:
answer = rag_simple("What is the profession of S R Rayhan? And what he is currently focusing on?", reg_retriever, llm)
print(answer)

Retrieving relevant documents for query: 'What is the profession of S R Rayhan? And what he is currently focusing on?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
Retrieved 2 relevant documents for the query: 'What is the profession of S R Rayhan? And what he is currently focusing on?'
My name is Md Shafikul Rahman Rayhan. In short S R Rayhan. I am currently working in ShellBeeHaken Ltd as an Associate Softwawre Engineer and currently learning RAG.

My name is Md Shafikul Rahman Rayhan. In short S R Rayhan. I am currently working in ShellBeeHaken Ltd as an Associate Softwawre Engineer and currently learning RAG.
The profession of S R Rayhan is Associate Software Engineer. 

He is currently focusing on learning RAG.


### Enhanced RAG Pipeline Features

In [162]:
def rag_advanced(query, retriever, llm, min_score = 0.2, top_K = 5, return_context = False):
    results = retriever.retrieve(query, top_k=top_K, score_threshold=min_score)

    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = '\n\n'.join(doc['content'] for doc in results)

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:200] + '...' if len(doc['content']) > 200 else doc['content']
    } for doc in results]

    confidence = max(doc['similarity_score'] for doc in results)

    ## Generate answer
    prompt = f"Use the following context to answer the question concisely:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output


In [164]:
result = rag_advanced("Who is currently working in ShellBeeHaken Ltd? Say the name.", reg_retriever, llm, min_score=0.1, top_K=3, return_context=True)

print("Answer :" , result['answer'])
print("\nSources :", result['sources'])
print("\nConfidence :", result['confidence'])
print("\nContext :", result['context'][:300])

Retrieving relevant documents for query: 'Who is currently working in ShellBeeHaken Ltd? Say the name.'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
Retrieved 3 relevant documents for the query: 'Who is currently working in ShellBeeHaken Ltd? Say the name.'
Answer : S R Rayhan (Md Shafikul Rahman Rayhan).

Sources : [{'source': '../data/text_files/myselft.txt', 'page': 'unknown', 'score': 0.11551743745803833, 'preview': 'My name is Md Shafikul Rahman Rayhan. In short S R Rayhan. I am currently working in ShellBeeHaken Ltd as an Associate Softwawre Engineer and currently learning RAG.'}, {'source': '../data/text_files/myself.txt', 'page': 'unknown', 'score': 0.11551743745803833, 'preview': 'My name is Md Shafikul Rahman Rayhan. In short S R Rayhan. I am currently working in ShellBeeHaken Ltd as an Associate Softwawre Engineer and currently learning RAG.'}, {'source': '../data/text_files/myself.txt', 'page': 'unknown', 'score': 0.11278402805328369, 'preview': 'My technical interests include machine learning, natural language processing, and building scalable systems. I am passionate about continuous